In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import *
set_style()

2026-04-03 01:20:47,631 - openket - INFO - openket v0.1.0 initialized successfully.


##### funciones auxiliares

In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean

In [4]:
def procesafile(at, lvl, folder, barrido="detuning_pa"):
    global detunings, Nss, non_converged, Probs, niveles, attrs
    detunings = []
    Probs = [[] for _ in np.arange(lvl**at)]
    Nss = []
    attrs = {}
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)
            
            detuning = f[dataset].attrs[barrido]
            tt = f[dataset].attrs['t']
            niveles = f[dataset].attrs['probs_label']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
    
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
            N_expect = rho[:, lvl**at] # la última columna es el valor medio de fotones en la cavidad
            N_expect = np.real(N_expect)

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            for i,ele in enumerate(lvl_prob):
                Probs[i].append(np.mean(ele[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
                
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]
    Probs = [np.real(np.array(p))[sorti] for p in Probs]

In [5]:
def picos(x,y):
    x = np.array(x)
    y = np.array(y)
    peaks, _ = find_peaks(y, prominence=0.001)

    if len(peaks) < 2:
        return None, None
    elif len(peaks) == 2:
        x1, x2 = x[peaks[0]], x[peaks[1]]
    else:
        # 3+ picos: tomar los dos extremos en x (ignorar el del medio)
        x1, x2 = x[peaks[0]], x[peaks[-1]]

    return x1, x2

## 1 átomo 3 niveles

#### CON rabi

In [16]:
at,lvl = 1,4
path = os.path.join("detunings",f"{at}at{lvl}lvl","conrabi")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1,x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss), h=0.003)
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.savefig(f"../figuras/1at3lvl_fotones.png")
plt.close()


plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    if j == 0: prob_e = prob
    
    plt.plot(detunings, prob, "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob_e)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob), h=0.003)
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.legend()
plt.savefig(f"../figuras/1at3lvl_excitado.png")
plt.close()
edist0 = x2-x1

In [7]:
attrs

{'detuning_ac': np.float64(0.0),
 'detuning_pa': np.float64(-3.2),
 'g': np.float64(3.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.01),
 'id': np.int32(19),
 'kappa': np.float64(1.0),
 'n': np.int32(5),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(2.0),
 'rabi_p': np.float64(0.1),
 't': array(['0', '15', '1000'], dtype=object)}

#### SIN rabi

In [15]:
at,lvl = 1,4
path = os.path.join("detunings",f"{at}at{lvl}lvl","sinrabi")

procesafile(at, lvl, path)

cut = 5
mask = (detunings >= -cut) & (detunings <= cut)
plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1,x2 = picos(detunings,Nss)
ax = plt.gca()
segmento(ax, x1, x2, y=max(Nss), h=0.0005)
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.005, ymax=0.01)
plt.savefig(f"../figuras/1at3lvl_fotones0.png")
plt.close()
fdist = x2-x1


plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    if j==0: prob_e = prob
    
    
    plt.plot(detunings[mask], prob[mask], "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
x1, x2 = picos(detunings,prob_e)
ax = plt.gca()
segmento(ax, x1, x2, y=max(prob_e), h=0.0005)
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.005, ymax=0.01)
plt.legend()
plt.savefig(f"../figuras/1at3lvl_excitado0.png")
plt.close()
edist = x2-x1

In [9]:
attrs

{'detuning_ac': np.float64(0.0),
 'detuning_pa': np.float64(-3.2),
 'g': np.float64(3.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.01),
 'id': np.int32(19),
 'kappa': np.float64(1.0),
 'n': np.int32(5),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(0.0),
 'rabi_p': np.float64(0.1),
 't': array(['0', '15', '1000'], dtype=object)}

### oscilaciones sin cavidad